# 🌑 Apollo Landing Site Thermal Model

## 1-D Lunar Thermal Simulation — Clean & Modular Version

This notebook simulates how temperature changes with **depth and time** in the lunar soil
(regolith) beneath Apollo landing sites.  It is organised so that:

- **All user settings live in one place** (Section 1 — Configuration).
- **Functions are in separate `.py` files** (the `lunar/` package) — easy to read and reuse.
- **The notebook just calls those functions** — keeping each cell short and focused.

### What you can do here

| Section | What it does |
|---|---|
| 1 — Configuration | Set location, density model, and physical parameters |
| 2 — Load DEM | Load the LOLA elevation map and extract terrain geometry |
| 3 — Run Model | Simulate temperature over several lunar days |
| 4 — Diurnal Cycles | Temperature vs time at different depths |
| 5 — Heatmap | 2-D depth × time view of the thermal wave |
| 6 — Apollo Comparison | Validate against Apollo 15/17 drill-hole measurements |
| **D — Single Location Analysis** | **Two-model comparison with 4-panel figure** |
| **E — Sensitivity Analysis** | **How does one parameter shift temperatures?** |
| **F — Material Property Profiles** | **Density, conductivity, thermal inertia vs depth** |
| **G — Sensitivity Tornado Chart** | **All-parameter importance ranking for articles** |
| **H — Apollo Residual Analysis** | **Publication-quality validation residual figures** |
| 10 — Batch Processing | Process many locations from a list | Process many locations from a list |

### Background: what is being modelled?

The Moon has no atmosphere, so its surface swings from ~100 K (night) to ~390 K (day)
over each 29.5-Earth-day lunar cycle.  Beneath the surface this swing damps out quickly:
by 1 m depth the temperature is nearly constant year-round.  This notebook solves the
1-D heat-conduction equation through the regolith to predict that depth profile.

**Apollo 15 (1971) and Apollo 17 (1972) drilled ~2 m into the surface** and measured
temperatures directly.  We compare our model predictions against those measurements.

---
*Package structure: `lunar/constants.py`, `models.py`, `dem.py`, `horizon.py`,
`solar.py`, `solver.py`, `analysis.py`, `plots.py`*


## 0. Imports

Run this cell first — it loads all modules and sets the plot style.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Local package — all the physics lives here
from lunar import constants, models, dem, horizon, solver, analysis, plots

print("✓ All modules loaded")
print(f"  Physical constants  : lunar.constants")
print(f"  Density models      : lunar.models")
print(f"  DEM loading         : lunar.dem")
print(f"  Horizon computation : lunar.horizon")
print(f"  Solar geometry      : lunar.solar")
print(f"  Thermal solver      : lunar.solver")
print(f"  Post-processing     : lunar.analysis")
print(f"  Plotting            : lunar.plots")


## 1. Configuration

**Edit only this cell** to change the simulation.  Everything else runs automatically.

### Quick-start recipes

| Scenario | Settings |
|---|---|
| Apollo 15 validation | `LAT=26.1323`, `LON=3.6285`, `MODEL='discrete'` |
| Apollo 17 validation | `LAT=20.1911`, `LON=30.7723`, `MODEL='discrete'` |
| Hayne 2017 global   | `MODEL='hayne_exponential'`, `H_PARAM=0.07` |
| Cold bias fix        | Increase `SUNSCALE` (try 1.10–1.15) |
| Hot bias fix         | Decrease `SUNSCALE` (try 0.95–1.00) |

### Parameter guide

| Parameter | Typical range | Effect |
|---|---|---|
| `SUNSCALE` | 0.95 – 1.15 | Higher → more solar energy → hotter surface |
| `ALBEDO` | 0.07 – 0.14 | Higher → more reflection → cooler surface |
| `EMISSIVITY` | 0.90 – 0.98 | Higher → better IR radiator → cooler surface |
| `CHI` | 1.5 – 4.0 | Controls how conductivity rises with temperature |
| `H_PARAM` | 0.03 – 0.15 m | Hayne scale height; larger = slower compaction |
| `NDAYS` | 2 – 10 | More days = better spin-up (3–5 is usually enough) |


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║             SIMULATION CONFIGURATION — EDIT HERE                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Location ──────────────────────────────────────────────────────────────────
LAT = 26.1323    # Target latitude  (degrees, positive = north)
LON =  3.6285    # Target longitude (degrees, 0–360 east)
# Apollo 15: LAT=26.1323, LON=3.6285
# Apollo 17: LAT=20.1911, LON=30.7723

# ── Density model ─────────────────────────────────────────────────────────────
MODEL = 'discrete'       # 'discrete' | 'hayne_exponential' | 'custom'
# 'discrete'          → sharp layers from Apollo drill cores  (recommended for Apollo sites)
# 'hayne_exponential' → smooth exponential from Diviner data  (good for other locations)
# 'custom'            → edit lunar/models.py: density_custom() / k_solid_custom()

H_PARAM = 0.07    # Hayne scale height OR discrete Layer-1 thickness (metres)
                  # 0.07 m = 7 cm  (Hayne et al. 2017 global average)
                  # Adjust this when running sensitivity analysis on H.

# ── Energy balance ────────────────────────────────────────────────────────────
SUNSCALE   = 1.10   # Solar flux multiplier (1.0 = nominal; 1.10 = 10% more solar energy)
ALBEDO     = 0.09   # Fraction of sunlight reflected (higher → cooler surface)
EMISSIVITY = 0.95   # IR emissivity (higher → better heat radiator → cooler)
CHI        = 2.7    # Radiative conductivity exponent (controls k ∝ T³ term)

# ── Simulation ────────────────────────────────────────────────────────────────
NDAYS  = 3          # Lunar days to simulate (first N-1 = spin-up, last = analysis)
T_INIT = 250.0      # Initial uniform temperature (K)

# ── Derived values (do not edit) ─────────────────────────────────────────────
MODEL_ID = models.MODEL_ID_MAP[MODEL]
models.set_hayne_h(H_PARAM)
models.set_layer1_h(H_PARAM)

print(f"\n{'═'*60}")
print("CONFIGURATION SUMMARY")
print(f"{'═'*60}")
print(f"  Location  : {LAT:.4f}°N, {LON:.4f}°E")
print(f"  Model     : {MODEL}  (model_id={MODEL_ID})")
print(f"  H-param   : {H_PARAM*100:.1f} cm")
print(f"  SUNSCALE  : {SUNSCALE}")
print(f"  ALBEDO    : {ALBEDO}")
print(f"  EMISSIVITY: {EMISSIVITY}")
print(f"  CHI       : {CHI}")
print(f"  Sim days  : {NDAYS}  ({NDAYS * 29.53:.1f} Earth days)")
print(f"{'═'*60}")


## 2. Load DEM and Extract Target Point

This cell loads the LOLA Digital Elevation Model (DEM) from the current directory
and snaps your target coordinates to the nearest pixel.

**Tip:** place `LDEM_*.LBL` and `LDEM_*.IMG` (or `.JP2`) next to this notebook.
Higher-resolution files are preferred automatically.


In [ ]:
# ── Load the elevation grid ────────────────────────────────────────────────────
ELEV_M, PIXEL_M, MAP_RES, ldem_label = dem.load_ldem()

# ── Extract the target point ──────────────────────────────────────────────────
(ROW, COL,
 ACTUAL_LAT, ACTUAL_LON,
 ELEVATION,
 SLOPE, ASPECT) = dem.extract_point(LAT, LON, ELEV_M, PIXEL_M, MAP_RES)

print(f"\n{'═'*60}")
print("TARGET POINT")
print(f"{'═'*60}")
print(f"  Requested : {LAT:.4f}°N, {LON:.4f}°E")
print(f"  Snapped to: {ACTUAL_LAT:.4f}°N, {ACTUAL_LON:.4f}°E")
print(f"  Pixel     : row={ROW}, col={COL}")
print(f"  Elevation : {ELEVATION:.1f} m")
print(f"  Slope     : {np.degrees(SLOPE):.2f}°")
print(f"  Aspect    : {np.degrees(ASPECT):.1f}°  (from north, clockwise)")
print(f"{'═'*60}")


In [ ]:
# ── Compute horizon profile (for topographic shadowing) ───────────────────────
from lunar.horizon import compute_horizon_profile, compute_sky_view_factor

N_AZ      = 360   # azimuth samples (360 = 1° resolution)
AZ_ANGLES = np.linspace(0, 2 * np.pi, N_AZ, endpoint=False, dtype=np.float32)

print("Computing horizon profile …")
HORIZONS = compute_horizon_profile(
    ROW, COL, ELEV_M, PIXEL_M, AZ_ANGLES, max_range_px=3000
)
SVF = compute_sky_view_factor(HORIZONS)

print(f"✓ Horizon computed  ({N_AZ} directions)")
print(f"  Sky-view factor : {SVF:.3f}  (1.0 = unobstructed flat terrain)")
print(f"  Max horizon     : {np.degrees(HORIZONS.max()):.1f}°")


## 3. Run the Thermal Model

This runs the simulation. Expect ~15–30 s for 3 lunar days.

In [ ]:
import time

Z_GRID = solver.create_depth_grid()
print(f"Depth grid: {len(Z_GRID)} points, 0 – {Z_GRID[-1]:.1f} m")
print(f"Running {NDAYS} lunar day(s) …")

t0 = time.time()
T_PROFILE, T_ARR = solver.solve_thermal_model(
    z_grid     = Z_GRID,
    T_init     = T_INIT,
    lat_deg    = ACTUAL_LAT,
    lon_deg    = ACTUAL_LON,
    slope      = SLOPE,
    aspect     = ASPECT,
    horizons   = HORIZONS,
    az_angles  = AZ_ANGLES,
    chi        = CHI,
    model_id   = MODEL_ID,
    sunscale   = SUNSCALE,
    ndays      = NDAYS,
    albedo     = ALBEDO,
    emissivity = EMISSIVITY,
)
print(f"✓ Done in {time.time()-t0:.1f} s")
print(f"  Shape: {T_PROFILE.shape[0]} snapshots × {T_PROFILE.shape[1]} depth nodes")
print(f"  T range: {T_PROFILE.min():.1f} – {T_PROFILE.max():.1f} K")

# Extract statistics for the final lunar day
STATS = analysis.extract_stats(T_PROFILE, T_ARR, Z_GRID)
CYCLES = analysis.get_diurnal_cycles(T_PROFILE, T_ARR, Z_GRID,
                                      depths_m=[0.0, 0.1, 0.5, 1.0, 2.0])

print(f"\nFinal-day surface temperatures:")
print(f"  Min : {STATS['T_min'][0]:.1f} K")
print(f"  Max : {STATS['T_max'][0]:.1f} K")
print(f"  Mean: {STATS['T_mean'][0]:.1f} K")


## 4. Diurnal Temperature Cycles

Temperature vs time at several depths over the last simulated lunar day.
The surface (0 cm) shows the extreme swing; deeper layers barely vary.


In [ ]:
fig = plots.diurnal_cycles(
    cycles     = CYCLES,
    lat        = ACTUAL_LAT,
    lon        = ACTUAL_LON,
    model_name = MODEL,
    sunscale   = SUNSCALE,
)
plt.show()


## 5. Temperature Heatmap — Depth × Time

A 2-D view of how the diurnal heat wave penetrates into the subsurface.
Red = hot (day), dark = cold (night).  The penetration depth is ~0.5 m.


In [ ]:
fig = plots.heatmap(
    T_profile  = T_PROFILE,
    t_arr      = T_ARR,
    z_grid     = Z_GRID,
    lat        = ACTUAL_LAT,
    lon        = ACTUAL_LON,
    model_name = MODEL,
    depth_limit= 1.5,       # show top 1.5 m only
    colormap   = 'hot',
)
plt.show()


## 6. Apollo Site Comparison

If your location is within 0.1° of Apollo 15 or Apollo 17, this cell
compares the model prediction with the actual drill-hole measurements.

**RMSE < 1 K** is an excellent fit.  If the model is too cold, increase
`SUNSCALE` or decrease `ALBEDO` in Section 1 and re-run Section 3.


In [ ]:
APOLLO_SITE = analysis.find_apollo_site(ACTUAL_LAT, ACTUAL_LON)

if APOLLO_SITE is None:
    print(f"ℹ  Not near an Apollo site.")
    print(f"   Current location: {ACTUAL_LAT:.3f}°N, {ACTUAL_LON:.3f}°E")
    print(f"   Apollo 15: 26.13°N, 3.63°E")
    print(f"   Apollo 17: 20.19°N, 30.77°E")
else:
    APOLLO_ERRORS = analysis.compute_apollo_errors(STATS['T_mean'], Z_GRID, APOLLO_SITE)
    print(f"✓ At {APOLLO_SITE} — RMSE={APOLLO_ERRORS['rmse']:.3f} K, "
          f"Bias={APOLLO_ERRORS['bias']:+.3f} K")

    fig = plots.apollo_comparison(
        stats      = STATS,
        errors     = APOLLO_ERRORS,
        site_name  = APOLLO_SITE,
        model_name = MODEL,
        sunscale   = SUNSCALE,
        chi        = CHI,
        albedo     = ALBEDO,
    )
    plt.show()


## D. Single Location Analysis

Run **two thermal models** at one or more locations and compare their temperature–depth profiles side-by-side.  A 4-panel figure is produced for each site:

| Panel | Content |
|---|---|
| Temperature Profiles | Mean ± diurnal range for Model A (blue) and B (red), plus Apollo HFE data if nearby |
| Diurnal Amplitude | (T_max − T_min) / 2 vs depth |
| Model Difference | B − A at each depth |
| Statistics | Parameter summary, RMSE / bias vs Apollo HFE |

**Edit `LOCATIONS_TO_RUN`, `SINGLE_MODEL_A/B`, and the per-model parameters below**, then run the cell.

Apollo reference coordinates:
- Apollo 15 — Hadley-Apennine: 26.1323 °N, 3.6285 °E
- Apollo 17 — Taurus-Littrow:  20.1911 °N, 30.7723 °E

In [ ]:
%matplotlib inline
# ============================================================================
# SECTION D: SINGLE LOCATION ANALYSIS
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec as _gs
import time
from tqdm.notebook import tqdm
from lunar import constants as _const
from lunar.models import density_hayne_py, k_solid_hayne_py, density_discrete_py, k_solid_discrete_py

# ── USER CONFIGURATION ───────────────────────────────────────────────────────
LOCATIONS_TO_RUN = [
    {'name': 'Apollo 15',  'lat': 26.1323, 'lon':  3.6285},
    {'name': 'Apollo 17',  'lat': 20.1911, 'lon': 30.7723},
    # Add more: {'name': 'My Site', 'lat': 0.0, 'lon': 0.0},
]
SINGLE_MODEL_A = 'hayne_exponential'   # shown in blue
SINGLE_MODEL_B = 'discrete'            # shown in red

MODEL_A_SUNSCALE   = 1.0;   MODEL_A_ALBEDO     = 0.09;  MODEL_A_CHI   = 2.7
MODEL_A_EMISSIVITY = 0.95;  MODEL_A_NDAYS      = 5;     MODEL_A_H     = 0.07

MODEL_B_SUNSCALE   = 1.0;   MODEL_B_ALBEDO     = 0.09;  MODEL_B_CHI   = 2.7
MODEL_B_EMISSIVITY = 0.95;  MODEL_B_NDAYS      = 5;     MODEL_B_H     = 0.07

# ↓ display units — change both together
DEPTH_SCALE      = 100;  DEPTH_UNIT_LABEL  = 'cm'     # ×1 for metres, ×100 for cm
TEMP_OFFSET      = 0;    TEMP_UNIT_LABEL   = 'K'      # -273.15 for °C
FIG_WIDTH        = 16;   FIG_HEIGHT        = 7
FONT_AXIS_LABEL  = 12;   FONT_TITLE        = 13;  FONT_SUPTITLE = 14
FONT_LEGEND      = 9;    FONT_STATS        = 9
APOLLO_MARKER    = 'o';  APOLLO_MARKER_SIZE = 25; APOLLO_MARKER_COLOR = 'green'
REPORT_DEPTHS_M  = [0.5, 1.0]   # depths shown in stats box

# ── HELPERS ──────────────────────────────────────────────────────────────────
def _haversine_m(lat1, lon1, lat2, lon2):
    R = 1_737_400.0
    la1, lo1, la2, lo2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((la2-la1)/2)**2 + np.cos(la1)*np.cos(la2)*np.sin((lo2-lo1)/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

def _run_model_d(model_key, H_val, sunscale, albedo, chi, emissivity, ndays,
                 alat, alon, sl, asp, horiz):
    mid = models.MODEL_ID_MAP[model_key]
    if model_key == 'hayne_exponential':
        TP, TA = solver.solve_with_h(
            Z_GRID, T_INIT, alat, alon, sl, asp, horiz, AZ_ANGLES,
            chi, mid, sunscale, ndays, H_param=H_val,
            albedo=albedo, emissivity=emissivity,
            density_fn=density_hayne_py, k_solid_fn=k_solid_hayne_py,
        )
    else:
        models.set_hayne_h(H_val); models.set_layer1_h(H_val)
        TP, TA = solver.solve_thermal_model(
            Z_GRID, T_INIT, alat, alon, sl, asp, horiz, AZ_ANGLES,
            chi, mid, sunscale, ndays, albedo=albedo, emissivity=emissivity,
        )
    return analysis.extract_stats(TP, TA, Z_GRID)

# ── MAIN LOOP ─────────────────────────────────────────────────────────────────
print(f"\n{'='*70}")
print(f"SINGLE LOCATION ANALYSIS — {SINGLE_MODEL_A.upper()} vs {SINGLE_MODEL_B.upper()}")
print(f"{'='*70}")

_STEPS = ['DEM + horizon', f'Model A', f'Model B', 'Plot']

for _loc in tqdm(LOCATIONS_TO_RUN, desc='Locations', unit='site'):
    SINGLE_LAT = _loc['lat']; SINGLE_LON = _loc['lon']; loc_name = _loc['name']
    print(f"\n{'─'*60}\n  {loc_name}  ({SINGLE_LAT:.4f}°N, {SINGLE_LON:.4f}°E)")
    try:
        with tqdm(total=4, desc=loc_name, leave=False,
                  bar_format='{desc}: {bar} {n_fmt}/{total_fmt}  [{elapsed}<{remaining}]') as pbar:

            # ── Step 1: DEM + horizon ──────────────────────────────────────
            pbar.set_description(f'{loc_name} › DEM + horizon')
            (i_t, j_t, act_lat, act_lon,
             elev_t, sl_t, asp_t) = dem.extract_point(SINGLE_LAT, SINGLE_LON,
                                                        ELEV_M, PIXEL_M, MAP_RES)
            snap_dist = _haversine_m(SINGLE_LAT, SINGLE_LON, act_lat, act_lon)
            horiz_t   = compute_horizon_profile(i_t, j_t, ELEV_M, PIXEL_M,
                                                AZ_ANGLES, max_range_px=1500)
            print(f"  DEM: ({act_lat:.5f}°N, {act_lon:.5f}°E)  snap={snap_dist:.0f} m")
            print(f"  Elev: {elev_t:.1f} m   Slope: {np.degrees(sl_t):.2f}°\n")
            pbar.update(1)

            # ── Step 2: Model A ────────────────────────────────────────────
            pbar.set_description(f'{loc_name} › Model A ({SINGLE_MODEL_A})')
            t0 = time.time()
            stats_A = _run_model_d(SINGLE_MODEL_A, MODEL_A_H, MODEL_A_SUNSCALE,
                                   MODEL_A_ALBEDO, MODEL_A_CHI, MODEL_A_EMISSIVITY,
                                   MODEL_A_NDAYS, act_lat, act_lon, sl_t, asp_t, horiz_t)
            print(f"  {SINGLE_MODEL_A} ✓ {time.time()-t0:.1f}s")
            pbar.update(1)

            # ── Step 3: Model B ────────────────────────────────────────────
            pbar.set_description(f'{loc_name} › Model B ({SINGLE_MODEL_B})')
            t0 = time.time()
            stats_B = _run_model_d(SINGLE_MODEL_B, MODEL_B_H, MODEL_B_SUNSCALE,
                                   MODEL_B_ALBEDO, MODEL_B_CHI, MODEL_B_EMISSIVITY,
                                   MODEL_B_NDAYS, act_lat, act_lon, sl_t, asp_t, horiz_t)
            print(f"  {SINGLE_MODEL_B} ✓ {time.time()-t0:.1f}s\n")
            pbar.update(1)

            # ── Step 4: Analysis + plot ────────────────────────────────────
            pbar.set_description(f'{loc_name} › Plotting')

            # Apollo match (generous 10 km radius)
            apollo_match = None; a_depths = a_temps = None
            for sn, si in _const.APOLLO_SITES.items():
                if _haversine_m(act_lat, act_lon, si['lat'], si['lon']) <= 10_000:
                    apollo_match = sn
                    d = _const.APOLLO_DATA[sn]
                    a_depths, a_temps = d['depths'], d['temps']
                    break

            err_A = (analysis.compute_apollo_errors(stats_A['T_mean'], Z_GRID, apollo_match)
                     if apollo_match else None)
            err_B = (analysis.compute_apollo_errors(stats_B['T_mean'], Z_GRID, apollo_match)
                     if apollo_match else None)
            if err_A:
                print(f"  {apollo_match}  A: RMSE={err_A['rmse']:.3f}K bias={err_A['bias']:+.3f}K  |  "
                      f"B: RMSE={err_B['rmse']:.3f}K bias={err_B['bias']:+.3f}K")

            # Unit conversions
            z_pl = Z_GRID * DEPTH_SCALE
            A_mn = stats_A['T_mean'] + TEMP_OFFSET
            A_mi = stats_A['T_min']  + TEMP_OFFSET
            A_mx = stats_A['T_max']  + TEMP_OFFSET
            B_mn = stats_B['T_mean'] + TEMP_OFFSET
            B_mi = stats_B['T_min']  + TEMP_OFFSET
            B_mx = stats_B['T_max']  + TEMP_OFFSET

            # 4-panel figure
            fig    = plt.figure(figsize=(FIG_WIDTH, FIG_HEIGHT))
            gs_obj = _gs.GridSpec(2, 4, figure=fig, hspace=0.12, height_ratios=[6, 1])
            ax1    = fig.add_subplot(gs_obj[0, 0])
            ax2    = fig.add_subplot(gs_obj[0, 1])
            ax3    = fig.add_subplot(gs_obj[0, 2])
            ax4    = fig.add_subplot(gs_obj[0, 3])
            ax_inf = fig.add_subplot(gs_obj[1, :])
            ax_inf.axis('off')

            lbl_A = 'A: ' + SINGLE_MODEL_A + (f' H={MODEL_A_H*100:.1f}cm' if SINGLE_MODEL_A == 'hayne_exponential' else '')
            lbl_B = 'B: ' + SINGLE_MODEL_B + (f' H={MODEL_B_H*100:.1f}cm' if SINGLE_MODEL_B == 'hayne_exponential' else '')

            # Panel 1: Temperature profiles
            ax1.fill_betweenx(z_pl, A_mi, A_mx, alpha=0.15, color='blue')
            ax1.plot(A_mn, z_pl, 'b-', lw=2.5, label=lbl_A)
            ax1.fill_betweenx(z_pl, B_mi, B_mx, alpha=0.15, color='red')
            ax1.plot(B_mn, z_pl, 'r-', lw=2.5, label=lbl_B)
            if a_depths is not None:
                ax1.scatter(a_temps + TEMP_OFFSET, a_depths * DEPTH_SCALE,
                            c=APOLLO_MARKER_COLOR, s=APOLLO_MARKER_SIZE, zorder=5,
                            marker=APOLLO_MARKER, label=f'{apollo_match} HFE',
                            edgecolors='darkgreen', linewidths=0.8)
            ax1.invert_yaxis()
            ax1.set_xlabel(f'Temperature ({TEMP_UNIT_LABEL})', fontsize=FONT_AXIS_LABEL, weight='bold')
            ax1.set_ylabel(f'Depth ({DEPTH_UNIT_LABEL})',       fontsize=FONT_AXIS_LABEL, weight='bold')
            ax1.set_title('Temperature Profiles',   fontsize=FONT_TITLE, weight='bold')
            ax1.legend(fontsize=FONT_LEGEND); ax1.grid(True, alpha=0.3)
            ax1.set_xlim(200, 300)

            # Panel 2: Diurnal amplitude
            A_amp = (stats_A['T_max'] - stats_A['T_min']) / 2.0
            B_amp = (stats_B['T_max'] - stats_B['T_min']) / 2.0
            ax2.plot(A_amp, z_pl, 'b-',  lw=2.5, label=f'A: {SINGLE_MODEL_A}')
            ax2.plot(B_amp, z_pl, 'r--', lw=2.5, label=f'B: {SINGLE_MODEL_B}')
            ax2.invert_yaxis()
            ax2.set_xlabel('Diurnal Amplitude (K)', fontsize=FONT_AXIS_LABEL, weight='bold')
            ax2.set_ylabel(f'Depth ({DEPTH_UNIT_LABEL})', fontsize=FONT_AXIS_LABEL, weight='bold')
            ax2.set_title('Diurnal Amplitude', fontsize=FONT_TITLE, weight='bold')
            ax2.legend(fontsize=FONT_LEGEND); ax2.grid(True, alpha=0.3)

            # Panel 3: Model difference B - A
            diff = stats_B['T_mean'] - stats_A['T_mean']
            ax3.plot(diff, z_pl, 'k-', lw=2.5)
            ax3.axvline(0, color='gray', ls='--', lw=1.5)
            ax3.invert_yaxis()
            ax3.set_xlabel('B - A  (K)', fontsize=FONT_AXIS_LABEL, weight='bold')
            ax3.set_ylabel(f'Depth ({DEPTH_UNIT_LABEL})', fontsize=FONT_AXIS_LABEL, weight='bold')
            ax3.set_title(f'{SINGLE_MODEL_B}\n- {SINGLE_MODEL_A}', fontsize=FONT_TITLE-1, weight='bold')
            ax3.grid(True, alpha=0.3)

            # Panel 4: Statistics
            ax4.axis('off')
            p  = "Parameters\n" + "─"*22 + "\n"
            p += f"Model A : {SINGLE_MODEL_A}\n"
            p += f"  sunscale={MODEL_A_SUNSCALE}  albedo={MODEL_A_ALBEDO}\n"
            p += f"  chi={MODEL_A_CHI}  emiss={MODEL_A_EMISSIVITY}\n"
            if SINGLE_MODEL_A == 'hayne_exponential':
                p += f"  H={MODEL_A_H*100:.1f} cm\n"
            p += f"Model B : {SINGLE_MODEL_B}\n"
            p += f"  sunscale={MODEL_B_SUNSCALE}  albedo={MODEL_B_ALBEDO}\n"
            p += f"  chi={MODEL_B_CHI}  emiss={MODEL_B_EMISSIVITY}\n"
            if SINGLE_MODEL_B == 'hayne_exponential':
                p += f"  H={MODEL_B_H*100:.1f} cm\n"
            elif SINGLE_MODEL_B == 'discrete':
                p += f"  Layer1_H={MODEL_B_H*100:.1f} cm\n"

            if a_depths is not None and err_A is not None:
                winner = 'A' if err_A['rmse'] < err_B['rmse'] else 'B'
                p += f"\nSite: {apollo_match}\n" + "─"*22 + "\n"
                p += f"     RMSE    Bias   T_surf\n"
                p += f"A  {err_A['rmse']:5.2f}K {err_A['bias']:+5.2f}K {stats_A['T_mean'][0]+TEMP_OFFSET:6.1f}{TEMP_UNIT_LABEL}\n"
                p += f"B  {err_B['rmse']:5.2f}K {err_B['bias']:+5.2f}K {stats_B['T_mean'][0]+TEMP_OFFSET:6.1f}{TEMP_UNIT_LABEL}\n"
                for d_m in REPORT_DEPTHS_M:
                    TA_d = np.interp(d_m, Z_GRID, stats_A['T_mean']) + TEMP_OFFSET
                    TB_d = np.interp(d_m, Z_GRID, stats_B['T_mean']) + TEMP_OFFSET
                    p += f"T@{d_m:.1f}m A={TA_d:.1f} B={TB_d:.1f}{TEMP_UNIT_LABEL}\n"
                p += f"\nBest fit: Model {winner}\n"
                p += f"dRMSE = {abs(err_A['rmse']-err_B['rmse']):.2f} K"
            else:
                p += f"\nNo Apollo site nearby\n"
                p += f"T_surf A = {stats_A['T_mean'][0]+TEMP_OFFSET:.1f} {TEMP_UNIT_LABEL}\n"
                p += f"T_surf B = {stats_B['T_mean'][0]+TEMP_OFFSET:.1f} {TEMP_UNIT_LABEL}"

            ax4.text(0.04, 0.98, p, transform=ax4.transAxes, fontsize=FONT_STATS,
                     va='top', family='monospace',
                     bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.6))
            ax4.set_title('Statistics', fontsize=FONT_TITLE, weight='bold')

            # Info bar
            ax_inf.text(0.5, 0.5,
                        f"DEM coords: lat={act_lat:.5f}N  lon={act_lon:.5f}E   "
                        f"(requested: {SINGLE_LAT:.5f}N, {SINGLE_LON:.5f}E  snap={snap_dist:.0f} m)",
                        transform=ax_inf.transAxes, fontsize=10, ha='center', va='center',
                        style='italic', bbox=dict(boxstyle='round', facecolor='lightcyan', alpha=0.7))

            fig.suptitle(
                f'Model Comparison: {SINGLE_MODEL_A.upper()} vs {SINGLE_MODEL_B.upper()}'
                + (f'  —  {apollo_match}' if apollo_match else f'  —  {loc_name}'),
                fontsize=FONT_SUPTITLE, fontweight='bold')
            plt.tight_layout(); plt.show()
            pbar.update(1)

        print(f"  ✓ {loc_name} done\n")

    except NameError as e:
        print(f"\n❌ Variable not found: {e}\n   Run cells 0–3 first.")
    except Exception as e:
        import traceback; print(f"\n❌ Error in {loc_name}: {e}"); traceback.print_exc()

models.set_hayne_h(H_PARAM); models.set_layer1_h(H_PARAM)
print(f"\n{'='*70}\nAll {len(LOCATIONS_TO_RUN)} location(s) processed.\n{'='*70}")

## E. Parameter Sensitivity Analysis

Vary **one model parameter** across a range while holding all others fixed.
Produces 3–4 panels showing how surface temperature, profile temperature,
diurnal amplitude, and (if near Apollo) RMSE change across the sweep range.

**Change `SENSITIVITY_PARAM`** and optionally `NDAYS_SENS` (1 day is fast
and adequate for relative sensitivity), then run.

| Parameter | What it controls |
|---|---|
| `sunscale` | Solar flux multiplier — accounts for dust or calibration offset |
| `albedo` | Fraction of sunlight reflected (higher → cooler surface) |
| `emissivity` | How efficiently the surface radiates IR heat |
| `chi` | Radiative conductivity exponent (k ∝ T³ term) |
| `h_parameter` | Hayne e-folding scale height (metres) |


In [ ]:
%matplotlib inline
# ============================================================================
# SECTION E: PARAMETER SENSITIVITY ANALYSIS
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
import time
from lunar import constants as _const

# ── USER CONFIGURATION ───────────────────────────────────────────────────────
SENSITIVITY_PARAM = 'h_parameter'   # ← CHANGE THIS

PARAM_RANGES = {
    'sunscale':    {'values': np.linspace(0.90, 1.15, 8), 'units': '(dimensionless)',
                    'description': 'Solar flux multiplier — 1.0 = nominal S0'},
    'albedo':      {'values': np.linspace(0.07, 0.14, 8), 'units': '(fraction)',
                    'description': 'Bond albedo — fraction of sunlight reflected'},
    'emissivity':  {'values': np.linspace(0.90, 0.98, 8), 'units': '(fraction)',
                    'description': 'IR emissivity — 1.0 = perfect blackbody'},
    'chi':         {'values': np.linspace(1.5,  4.0,  8), 'units': '(dimensionless)',
                    'description': 'Radiative conductivity exponent chi'},
    'h_parameter': {'values': np.linspace(0.03, 0.15, 8), 'units': '(metres)',
                    'description': 'Hayne e-folding scale height'},
}

# Location — re-uses SINGLE_LAT/LON if Section D was run, else falls back to main
_loc_lat = SINGLE_LAT if 'SINGLE_LAT' in dir() else ACTUAL_LAT
_loc_lon = SINGLE_LON if 'SINGLE_LON' in dir() else ACTUAL_LON
NDAYS_SENS = 1   # 1 day is fast and adequate for relative sensitivity

# ── EXECUTION ─────────────────────────────────────────────────────────────────
if SENSITIVITY_PARAM not in PARAM_RANGES:
    print(f"Unknown param '{SENSITIVITY_PARAM}'. Valid: {list(PARAM_RANGES.keys())}")
else:
    cfg    = PARAM_RANGES[SENSITIVITY_PARAM]
    pvals  = cfg['values']
    p_base = (SUNSCALE    if SENSITIVITY_PARAM == 'sunscale'   else
              ALBEDO      if SENSITIVITY_PARAM == 'albedo'     else
              EMISSIVITY  if SENSITIVITY_PARAM == 'emissivity' else
              CHI         if SENSITIVITY_PARAM == 'chi'        else H_PARAM)

    print(f"\n{'='*70}")
    print(f"SENSITIVITY — {SENSITIVITY_PARAM.upper()}")
    print(f"{'='*70}")
    print(f"  {cfg['description']}")
    print(f"  Range    : {pvals[0]:.4f} -> {pvals[-1]:.4f}  ({len(pvals)} values)")
    print(f"  Baseline : {p_base:.4f}")
    print(f"  Location : {_loc_lat:.4f}N, {_loc_lon:.4f}E\n")

    (i_s, j_s, act_lat_s, act_lon_s,
     _, sl_s, asp_s) = dem.extract_point(_loc_lat, _loc_lon, ELEV_M, PIXEL_M, MAP_RES)
    horiz_s = compute_horizon_profile(i_s, j_s, ELEV_M, PIXEL_M, AZ_ANGLES, max_range_px=1500)

    apollo_s = None
    for sn, si in _const.APOLLO_SITES.items():
        if abs(act_lat_s - si['lat']) < 0.15 and abs(act_lon_s - si['lon']) < 0.15:
            apollo_s = sn
            print(f"  Apollo site: {apollo_s} — will compute RMSE\n")
            break
    if not apollo_s:
        print("  No Apollo site nearby — sensitivity plots only\n")

    t_start = time.time()
    sens_results = analysis.run_sensitivity(
        param_name   = SENSITIVITY_PARAM,
        param_values = pvals,
        z_grid       = Z_GRID,
        T_init       = T_INIT,
        lat_deg      = act_lat_s,
        lon_deg      = act_lon_s,
        slope        = sl_s,
        aspect       = asp_s,
        horizons     = horiz_s,
        az_angles    = AZ_ANGLES,
        chi          = CHI,
        model_id     = MODEL_ID,
        sunscale     = SUNSCALE,
        ndays        = NDAYS_SENS,
        albedo       = ALBEDO,
        emissivity   = EMISSIVITY,
        baseline_h   = H_PARAM,
        apollo_site  = apollo_s,
        verbose      = True,
    )
    print(f"\n✓ Sweep complete in {time.time()-t_start:.1f} s")

    out_T_surf = np.array([r['stats']['T_mean'][0]       for r in sens_results])
    out_T_mean = np.array([np.mean(r['stats']['T_mean']) for r in sens_results])
    out_T_amp  = np.array([r['stats']['T_amplitude'][0]  for r in sens_results])
    out_rmse   = np.array([r['errors']['rmse'] if r['errors'] else np.nan
                           for r in sens_results])

    # ── PLOTTING ──────────────────────────────────────────────────────────────
    has_rmse  = apollo_s is not None and not np.all(np.isnan(out_rmse))
    n_panels  = 4 if has_rmse else 3
    fig, axes = plt.subplots(1, n_panels, figsize=(5*n_panels, 5))
    MK = 'o-'

    axes[0].plot(pvals, out_T_surf, MK, color='crimson', lw=2, ms=8)
    axes[0].axvline(p_base, color='gray', ls='--', lw=1.5, label=f'Baseline: {p_base:.4f}')
    axes[0].set_xlabel(f'{SENSITIVITY_PARAM}  {cfg["units"]}', fontsize=11, weight='bold')
    axes[0].set_ylabel('Mean Surface Temperature (K)', fontsize=11, weight='bold')
    axes[0].set_title('Surface Temperature', fontsize=12, weight='bold')
    axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

    axes[1].plot(pvals, out_T_mean, MK, color='royalblue', lw=2, ms=8)
    axes[1].axvline(p_base, color='gray', ls='--', lw=1.5)
    axes[1].set_xlabel(f'{SENSITIVITY_PARAM}  {cfg["units"]}', fontsize=11, weight='bold')
    axes[1].set_ylabel('Profile-Avg Mean Temp (K)', fontsize=11, weight='bold')
    axes[1].set_title('Profile Temperature', fontsize=12, weight='bold')
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(pvals, out_T_amp, MK, color='darkorange', lw=2, ms=8)
    axes[2].axvline(p_base, color='gray', ls='--', lw=1.5)
    axes[2].set_xlabel(f'{SENSITIVITY_PARAM}  {cfg["units"]}', fontsize=11, weight='bold')
    axes[2].set_ylabel('Surface Diurnal Amplitude (K)', fontsize=11, weight='bold')
    axes[2].set_title('Diurnal Amplitude', fontsize=12, weight='bold')
    axes[2].grid(True, alpha=0.3)

    if has_rmse:
        best_idx  = int(np.nanargmin(out_rmse))
        best_pval = pvals[best_idx]
        axes[3].plot(pvals, out_rmse, MK, color='purple', lw=2, ms=8, label='RMSE')
        axes[3].axvline(best_pval, color='red', ls='--', lw=2,
                        label=f'Best: {best_pval:.4f}')
        axes[3].scatter([best_pval], [out_rmse[best_idx]], color='red', s=120, zorder=5)
        axes[3].axvline(p_base, color='gray', ls=':', lw=1.5, label=f'Baseline: {p_base:.4f}')
        axes[3].set_xlabel(f'{SENSITIVITY_PARAM}  {cfg["units"]}', fontsize=11, weight='bold')
        axes[3].set_ylabel(f'RMSE vs {apollo_s} HFE (K)', fontsize=11, weight='bold')
        axes[3].set_title('Validation Accuracy', fontsize=12, weight='bold')
        axes[3].legend(fontsize=9); axes[3].grid(True, alpha=0.3)
        print(f"\n  Best-fit {SENSITIVITY_PARAM} = {best_pval:.4f} {cfg['units']}")
        print(f"  Minimum RMSE = {out_rmse[best_idx]:.3f} K  (vs {apollo_s} HFE)")

    fig.suptitle(
        f'Sensitivity: {SENSITIVITY_PARAM.upper()}  —  {cfg["description"]}\n'
        f'Location: {act_lat_s:.3f}N, {act_lon_s:.3f}E'
        + (f'  |  {apollo_s}' if apollo_s else ''),
        fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout(); plt.show()

    dT = out_T_surf.max() - out_T_surf.min()
    level = ("VERY HIGH (>50 K)" if dT > 50 else "HIGH (>20 K)"   if dT > 20 else
             "MODERATE (>5 K)"  if dT > 5  else "LOW (<5 K)")
    print(f"\n{'─'*60}")
    print(f"SENSITIVITY SUMMARY: {SENSITIVITY_PARAM.upper()}")
    print(f"  T_surf range  : {out_T_surf.min():.1f} - {out_T_surf.max():.1f} K  (delta={dT:.1f} K)")
    print(f"  Profile mean  : {out_T_mean.min():.1f} - {out_T_mean.max():.1f} K")
    print(f"  Amplitude     : {out_T_amp.min():.1f} - {out_T_amp.max():.1f} K")
    print(f"  Sensitivity   : {level}\n{'─'*60}")


## F. Material Property Profiles  *(Article Figure)*

Visualise how **bulk density**, **thermal conductivity**, and **thermal inertia**
vary with depth for the two density models.  This figure belongs in the
*Methods* or *Model Description* section of an article — it explains the
physical differences between the models before showing their temperature predictions.

- **Panel 1** Bulk density ρ(z) — Hayne smooth vs Discrete step-wise
- **Panel 2** Solid (contact) conductivity k_solid(z)
- **Panel 3** Total conductivity k(T,z) = k_solid · (1 + χ(T/T_ref)³) at two temperatures
- **Panel 4** Thermal inertia TI = √(ρ c_p k_solid) — controls how quickly a surface heats/cools


In [ ]:
%matplotlib inline
# ============================================================================
# SECTION F: MATERIAL PROPERTY PROFILES
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
from lunar.models import (density_hayne_py, k_solid_hayne_py,
                          density_discrete_py, k_solid_discrete_py,
                          heat_capacity)

z    = np.linspace(0, 1.5, 500)   # depth 0 – 1.5 m
z_cm = z * 100

rho_H = np.array([density_hayne_py(zi)    for zi in z])
rho_D = np.array([density_discrete_py(zi) for zi in z])
ks_H  = np.array([k_solid_hayne_py(zi)    for zi in z])
ks_D  = np.array([k_solid_discrete_py(zi) for zi in z])

chi_val = CHI if 'CHI' in dir() else 2.7
T_ref   = 350.0
T_warm  = 300.0   # typical dayside surface
T_cold  = 250.0   # typical subsurface

k_H_warm = ks_H * (1 + chi_val * (T_warm / T_ref)**3)
k_H_cold = ks_H * (1 + chi_val * (T_cold / T_ref)**3)
k_D_warm = ks_D * (1 + chi_val * (T_warm / T_ref)**3)
k_D_cold = ks_D * (1 + chi_val * (T_cold / T_ref)**3)

cp_mid = heat_capacity(np.float32(250.0))   # J kg-1 K-1 at ~250 K
TI_H   = np.sqrt(rho_H * cp_mid * ks_H)
TI_D   = np.sqrt(rho_D * cp_mid * ks_D)

fig, axes = plt.subplots(1, 4, figsize=(18, 6))

# Panel 1: Density
axes[0].plot(rho_H, z_cm, 'b-',  lw=2.5, label='Hayne 2017 (exponential)')
axes[0].plot(rho_D, z_cm, 'r--', lw=2.5, label='Discrete layers')
axes[0].axhline(7,  color='blue',  ls=':', lw=1.2, alpha=0.7, label='H = 7 cm (Hayne)')
axes[0].axhline(20, color='red',   ls='-.', lw=1.2, alpha=0.7, label='L2 boundary = 20 cm')
axes[0].invert_yaxis()
axes[0].set_xlabel('Bulk Density (kg m$^{-3}$)', fontsize=12, weight='bold')
axes[0].set_ylabel('Depth (cm)',                  fontsize=12, weight='bold')
axes[0].set_title('Density Profile',              fontsize=13, weight='bold')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# Panel 2: Solid conductivity
axes[1].plot(ks_H * 1e3, z_cm, 'b-',  lw=2.5, label='Hayne 2017')
axes[1].plot(ks_D * 1e3, z_cm, 'r--', lw=2.5, label='Discrete layers')
axes[1].invert_yaxis()
axes[1].set_xlabel('$k_{solid}$ (mW m$^{-1}$ K$^{-1}$)', fontsize=12, weight='bold')
axes[1].set_ylabel('Depth (cm)',                           fontsize=12, weight='bold')
axes[1].set_title('Solid Conductivity',                    fontsize=13, weight='bold')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

# Panel 3: Total conductivity at two temperatures
axes[2].plot(k_H_warm * 1e3, z_cm, 'b-',  lw=2.5, label=f'Hayne  T={T_warm:.0f} K')
axes[2].plot(k_H_cold * 1e3, z_cm, 'b--', lw=1.5, label=f'Hayne  T={T_cold:.0f} K', alpha=0.7)
axes[2].plot(k_D_warm * 1e3, z_cm, 'r-',  lw=2.5, label=f'Discr. T={T_warm:.0f} K')
axes[2].plot(k_D_cold * 1e3, z_cm, 'r--', lw=1.5, label=f'Discr. T={T_cold:.0f} K', alpha=0.7)
axes[2].invert_yaxis()
axes[2].set_xlabel('Total $k$ (mW m$^{-1}$ K$^{-1}$)', fontsize=12, weight='bold')
axes[2].set_ylabel('Depth (cm)',                          fontsize=12, weight='bold')
axes[2].set_title('Total Conductivity',                   fontsize=13, weight='bold')
axes[2].legend(fontsize=8); axes[2].grid(True, alpha=0.3)

# Panel 4: Thermal inertia
axes[3].plot(TI_H, z_cm, 'b-',  lw=2.5, label='Hayne 2017')
axes[3].plot(TI_D, z_cm, 'r--', lw=2.5, label='Discrete layers')
axes[3].axvline(50,  color='gray', ls=':',  lw=1.2, label='TI=50 (loose dust)')
axes[3].axvline(200, color='gray', ls='-.', lw=1.2, label='TI=200 (compacted rock)')
axes[3].invert_yaxis()
axes[3].set_xlabel('Thermal Inertia (tiu)', fontsize=12, weight='bold')
axes[3].set_ylabel('Depth (cm)',             fontsize=12, weight='bold')
axes[3].set_title('Thermal Inertia',         fontsize=13, weight='bold')
axes[3].legend(fontsize=8); axes[3].grid(True, alpha=0.3)

fig.suptitle(
    f'Material Property Profiles — Hayne 2017 vs Discrete Layer Model\n'
    f'Depth 0 – 150 cm   |   $\\chi$ = {chi_val:.1f}',
    fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

iz1m = np.argmin(np.abs(z - 1.0))
print(f"Surface thermal inertia:   Hayne = {TI_H[0]:.1f} tiu   Discrete = {TI_D[0]:.1f} tiu")
print(f"1 m depth thermal inertia: Hayne = {TI_H[iz1m]:.1f} tiu   Discrete = {TI_D[iz1m]:.1f} tiu")


## G. Sensitivity Tornado Chart  *(Article Figure)*

Sweeps **all five model parameters** and ranks them by their total influence
on mean surface temperature.  The tornado (horizontal bar) chart is a standard
figure in sensitivity studies — it lets readers immediately see *which* parameter
matters most, without reading through five separate sweep plots.

A second panel shows how RMSE vs Apollo HFE data changes over each parameter
range, so the chart doubles as a parameter-importance ranking for *validation* too.

> **Runtime:** sweeps 5 × 8 = 40 simulations.  With `NDAYS_TORNADO = 1` expect
> roughly 3–8 minutes (most time is the h_parameter pure-Python runs).


In [ ]:
%matplotlib inline
# ============================================================================
# SECTION G: SENSITIVITY TORNADO CHART
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
import time
from lunar import constants as _const

# ── USER CONFIGURATION ───────────────────────────────────────────────────────
NDAYS_TORNADO = 1   # 1 day is fast; set to 3 for publication accuracy

TORNADO_RANGES = {
    'sunscale':    np.linspace(0.90, 1.15, 8),
    'albedo':      np.linspace(0.07, 0.14, 8),
    'emissivity':  np.linspace(0.90, 0.98, 8),
    'chi':         np.linspace(1.5,  4.0,  8),
    'h_parameter': np.linspace(0.03, 0.15, 8),
}
PARAM_LABELS = {
    'sunscale':    'Solar flux\nmultiplier',
    'albedo':      'Bond\nalbedo',
    'emissivity':  'IR\nemissivity',
    'chi':         'Radiative k\nexponent chi',
    'h_parameter': 'Scale height\nH (m)',
}

# ── EXTRACT GEOMETRY AT MAIN CONFIG LOCATION ─────────────────────────────────
(i_g, j_g, _alat_g, _alon_g, _, _sl_g, _asp_g) = dem.extract_point(
    ACTUAL_LAT, ACTUAL_LON, ELEV_M, PIXEL_M, MAP_RES)
_horiz_g = compute_horizon_profile(i_g, j_g, ELEV_M, PIXEL_M, AZ_ANGLES, max_range_px=1500)
_apollo_g = analysis.find_apollo_site(_alat_g, _alon_g)

# ── RUN ALL SWEEPS ────────────────────────────────────────────────────────────
tornado_data = {}
t_all = time.time()

for param, pvals in TORNADO_RANGES.items():
    print(f"  Sweeping {param} ...", end=' ', flush=True)
    t0  = time.time()
    res = analysis.run_sensitivity(
        param_name=param, param_values=pvals,
        z_grid=Z_GRID, T_init=T_INIT,
        lat_deg=_alat_g, lon_deg=_alon_g,
        slope=_sl_g, aspect=_asp_g,
        horizons=_horiz_g, az_angles=AZ_ANGLES,
        chi=CHI, model_id=MODEL_ID, sunscale=SUNSCALE,
        ndays=NDAYS_TORNADO,
        albedo=ALBEDO, emissivity=EMISSIVITY,
        baseline_h=H_PARAM, apollo_site=_apollo_g, verbose=False,
    )
    T_surfs = np.array([r['stats']['T_mean'][0] for r in res])
    rmses   = np.array([r['errors']['rmse'] if r['errors'] else np.nan for r in res])
    tornado_data[param] = {
        'dT':    float(T_surfs.max() - T_surfs.min()),
        'dRMSE': float(np.nanmax(rmses) - np.nanmin(rmses))
                 if not np.all(np.isnan(rmses)) else np.nan,
    }
    print(f"dT={tornado_data[param]['dT']:.1f} K  ({time.time()-t0:.0f}s)")

print(f"\n✓ All sweeps done in {time.time()-t_all:.0f} s")

# ── PLOTTING ──────────────────────────────────────────────────────────────────
params_sorted = sorted(tornado_data, key=lambda p: tornado_data[p]['dT'], reverse=True)
has_rmse_g    = not all(np.isnan(tornado_data[p]['dRMSE']) for p in params_sorted)
n_panels      = 2 if has_rmse_g else 1
fig, axes     = plt.subplots(1, n_panels, figsize=(7*n_panels, 5))
if n_panels == 1:
    axes = [axes]

colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(params_sorted)))
y_pos  = np.arange(len(params_sorted))

# Panel 1: Surface temperature range
bars = axes[0].barh(y_pos,
                    [tornado_data[p]['dT'] for p in params_sorted],
                    color=colors, edgecolor='black', linewidth=0.7)
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels([PARAM_LABELS.get(p, p) for p in params_sorted], fontsize=11)
axes[0].set_xlabel('Surface Temperature Range  ΔT (K)', fontsize=12, weight='bold')
axes[0].set_title('Parameter Sensitivity — Surface Temperature', fontsize=13, weight='bold')
axes[0].grid(True, axis='x', alpha=0.3)
for bar, p in zip(bars, params_sorted):
    axes[0].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                 f"{tornado_data[p]['dT']:.1f} K", va='center', fontsize=10)

# Panel 2: RMSE range
if has_rmse_g:
    dRMSE_vals = [tornado_data[p]['dRMSE'] if not np.isnan(tornado_data[p]['dRMSE']) else 0.0
                  for p in params_sorted]
    bars2 = axes[1].barh(y_pos, dRMSE_vals, color=colors, edgecolor='black', linewidth=0.7)
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels([PARAM_LABELS.get(p, p) for p in params_sorted], fontsize=11)
    axes[1].set_xlabel(f'RMSE Range vs {_apollo_g} HFE  (K)', fontsize=12, weight='bold')
    axes[1].set_title('Parameter Sensitivity — Validation RMSE', fontsize=13, weight='bold')
    axes[1].grid(True, axis='x', alpha=0.3)
    for bar, v in zip(bars2, dRMSE_vals):
        if v > 0:
            axes[1].text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                         f"{v:.2f} K", va='center', fontsize=10)

fig.suptitle(
    f'Tornado Chart — Parameter Influence on Lunar Thermal Model\n'
    f'Location: {_alat_g:.3f}N, {_alon_g:.3f}E  |  Model: {MODEL}',
    fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print("\nRanking (most to least influential on surface temperature):")
for rank, p in enumerate(params_sorted, 1):
    print(f"  {rank}. {p:<14}  ΔT = {tornado_data[p]['dT']:.1f} K")


## H. Apollo HFE Validation — Residual Analysis  *(Article Figure)*

Runs both models at Apollo 15 and 17 and produces publication-quality
validation figures:

- **Row 1** Temperature profiles (model A, model B, Apollo HFE data points)
- **Row 2** Residuals — (model − Apollo) at each HFE probe depth

The residual plot is the key *quantitative validation* figure for an article:
it shows *where* in the column each model agrees or disagrees with observations.
A perfect model would have all points on the zero line.

Edit `RESIDUAL_MODEL_A/B` and the per-model parameters to test different
configurations before submission.


In [ ]:
%matplotlib inline
# ============================================================================
# SECTION H: APOLLO HFE VALIDATION — RESIDUAL ANALYSIS
# ============================================================================
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import time
from lunar import constants as _const
from lunar.models import density_hayne_py, k_solid_hayne_py

# ── USER CONFIGURATION ───────────────────────────────────────────────────────
RESIDUAL_MODEL_A    = 'hayne_exponential'
RESIDUAL_MODEL_B    = 'discrete'
RESIDUAL_NDAYS      = 5      # more days = better spin-up
RESIDUAL_SUNSCALE   = SUNSCALE
RESIDUAL_ALBEDO     = ALBEDO
RESIDUAL_CHI        = CHI
RESIDUAL_EMISSIVITY = EMISSIVITY
RESIDUAL_H_A        = 0.07   # metres
RESIDUAL_H_B        = 0.07
APOLLO_SITES_TO_RUN = ['Apollo 15', 'Apollo 17']

# ── EXECUTION ─────────────────────────────────────────────────────────────────
print(f"\n{'='*65}\nAPOLLO RESIDUAL ANALYSIS\n{'='*65}\n")

site_results = {}

def _run_res(model_key, H_val, sunscale, albedo, chi, emissivity, ndays,
             alat, alon, sl, asp, horiz):
    mid = models.MODEL_ID_MAP[model_key]
    if model_key == 'hayne_exponential':
        TP, TA = solver.solve_with_h(
            Z_GRID, T_INIT, alat, alon, sl, asp, horiz, AZ_ANGLES,
            chi, mid, sunscale, ndays, H_param=H_val,
            albedo=albedo, emissivity=emissivity,
            density_fn=density_hayne_py, k_solid_fn=k_solid_hayne_py,
        )
    else:
        models.set_hayne_h(H_val); models.set_layer1_h(H_val)
        TP, TA = solver.solve_thermal_model(
            Z_GRID, T_INIT, alat, alon, sl, asp, horiz, AZ_ANGLES,
            chi, mid, sunscale, ndays, albedo=albedo, emissivity=emissivity,
        )
    return analysis.extract_stats(TP, TA, Z_GRID)

for site_name in APOLLO_SITES_TO_RUN:
    if site_name not in _const.APOLLO_SITES:
        print(f"  {site_name} not found — skipping"); continue
    si = _const.APOLLO_SITES[site_name]
    print(f"{'─'*50}\n{site_name}: ({si['lat']:.4f}N, {si['lon']:.4f}E)")

    (i_r, j_r, alat, alon, _, sl_r, asp_r) = dem.extract_point(
        si['lat'], si['lon'], ELEV_M, PIXEL_M, MAP_RES)
    horiz_r = compute_horizon_profile(i_r, j_r, ELEV_M, PIXEL_M, AZ_ANGLES, max_range_px=1500)

    print(f"  Running {RESIDUAL_MODEL_A} ...", end=' ', flush=True)
    t0 = time.time()
    stats_A = _run_res(RESIDUAL_MODEL_A, RESIDUAL_H_A, RESIDUAL_SUNSCALE,
                       RESIDUAL_ALBEDO, RESIDUAL_CHI, RESIDUAL_EMISSIVITY,
                       RESIDUAL_NDAYS, alat, alon, sl_r, asp_r, horiz_r)
    err_A   = analysis.compute_apollo_errors(stats_A['T_mean'], Z_GRID, site_name)
    print(f"✓ {time.time()-t0:.1f}s  RMSE={err_A['rmse']:.3f} K")

    print(f"  Running {RESIDUAL_MODEL_B} ...", end=' ', flush=True)
    t0 = time.time()
    stats_B = _run_res(RESIDUAL_MODEL_B, RESIDUAL_H_B, RESIDUAL_SUNSCALE,
                       RESIDUAL_ALBEDO, RESIDUAL_CHI, RESIDUAL_EMISSIVITY,
                       RESIDUAL_NDAYS, alat, alon, sl_r, asp_r, horiz_r)
    err_B   = analysis.compute_apollo_errors(stats_B['T_mean'], Z_GRID, site_name)
    print(f"✓ {time.time()-t0:.1f}s  RMSE={err_B['rmse']:.3f} K\n")

    site_results[site_name] = dict(stats_A=stats_A, stats_B=stats_B,
                                    err_A=err_A, err_B=err_B)

models.set_hayne_h(H_PARAM); models.set_layer1_h(H_PARAM)

# ── PLOTTING ──────────────────────────────────────────────────────────────────
n_sites = len(site_results)
if n_sites == 0:
    print("No sites processed.")
else:
    fig = plt.figure(figsize=(7*n_sites, 12))
    gs  = GridSpec(2, n_sites, figure=fig, hspace=0.38, wspace=0.32)

    for col, (site_name, R) in enumerate(site_results.items()):
        z_cm     = Z_GRID * 100
        a_dep_cm = R['err_A']['apollo_depths'] * 100
        a_temps  = R['err_A']['apollo_temps']

        # Row 0: temperature profiles
        ax0 = fig.add_subplot(gs[0, col])
        ax0.fill_betweenx(z_cm, R['stats_A']['T_min'], R['stats_A']['T_max'],
                          alpha=0.12, color='blue')
        ax0.plot(R['stats_A']['T_mean'], z_cm, 'b-', lw=2.5,
                 label=f'A: {RESIDUAL_MODEL_A}\nRMSE={R["err_A"]["rmse"]:.2f} K')
        ax0.fill_betweenx(z_cm, R['stats_B']['T_min'], R['stats_B']['T_max'],
                          alpha=0.12, color='red')
        ax0.plot(R['stats_B']['T_mean'], z_cm, 'r-', lw=2.5,
                 label=f'B: {RESIDUAL_MODEL_B}\nRMSE={R["err_B"]["rmse"]:.2f} K')
        ax0.scatter(a_temps, a_dep_cm, c='green', s=55, zorder=6, marker='o',
                    label=f'{site_name} HFE', edgecolors='darkgreen', linewidths=0.8)
        ax0.invert_yaxis()
        ax0.set_xlabel('Temperature (K)', fontsize=12, weight='bold')
        ax0.set_ylabel('Depth (cm)',       fontsize=12, weight='bold')
        ax0.set_title(f'{site_name}\nTemperature Profiles', fontsize=13, weight='bold')
        ax0.legend(fontsize=8, loc='lower right'); ax0.grid(True, alpha=0.3)
        ax0.set_xlim(240, 270)

        # Row 1: residuals
        ax1 = fig.add_subplot(gs[1, col])
        res_A = R['err_A']['residuals']
        res_B = R['err_B']['residuals']
        ax1.axvline(0, color='black', lw=1.5, ls='--', label='Zero residual')
        ax1.axvspan(-1, 1, alpha=0.06, color='green')
        ax1.scatter(res_A, a_dep_cm, c='blue', s=80, zorder=5,
                    marker='o', label=f'A: {RESIDUAL_MODEL_A}',
                    edgecolors='navy', linewidths=0.8)
        ax1.scatter(res_B, a_dep_cm, c='red',  s=80, zorder=5,
                    marker='s', label=f'B: {RESIDUAL_MODEL_B}',
                    edgecolors='darkred', linewidths=0.8)
        ax1.plot(res_A, a_dep_cm, 'b-', lw=1.5, alpha=0.6)
        ax1.plot(res_B, a_dep_cm, 'r-', lw=1.5, alpha=0.6)
        ax1.invert_yaxis()
        ax1.set_xlabel('Residual: Model - Apollo HFE (K)', fontsize=12, weight='bold')
        ax1.set_ylabel('Depth (cm)',                        fontsize=12, weight='bold')
        ax1.set_title(f'{site_name}\nModel Residuals', fontsize=13, weight='bold')
        ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)
        txt = (f"A: RMSE={R['err_A']['rmse']:.2f}K  bias={R['err_A']['bias']:+.2f}K\n"
               f"B: RMSE={R['err_B']['rmse']:.2f}K  bias={R['err_B']['bias']:+.2f}K")
        ax1.text(0.97, 0.04, txt, transform=ax1.transAxes, fontsize=8.5,
                 va='bottom', ha='right', family='monospace',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

    fig.suptitle(
        f'Apollo HFE Validation — '
        f'{RESIDUAL_MODEL_A.upper()} vs {RESIDUAL_MODEL_B.upper()}\n'
        f'SUNSCALE={RESIDUAL_SUNSCALE}  ALBEDO={RESIDUAL_ALBEDO}  chi={RESIDUAL_CHI}  '
        f'H_A={RESIDUAL_H_A*100:.0f}cm  H_B={RESIDUAL_H_B*100:.0f}cm',
        fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

    print(f"\n{'─'*65}\nVALIDATION SUMMARY")
    print(f"{'─'*65}")
    print(f"{'Site':<12}  {'Model':<24}  {'RMSE':>6}  {'Bias':>7}  {'MAE':>6}")
    print("─" * 65)
    for site_name, R in site_results.items():
        for lbl, err in [('A: '+RESIDUAL_MODEL_A[:20], R['err_A']),
                         ('B: '+RESIDUAL_MODEL_B[:20], R['err_B'])]:
            print(f"{site_name:<12}  {lbl:<24}  "
                  f"{err['rmse']:>5.2f}K  {err['bias']:>+6.2f}K  {err['mae']:>5.2f}K")
    print(f"{'─'*65}")


## 10. Batch Processing

Process a list of locations automatically.  Results are collected in
`BATCH_RESULTS` (a list of dicts) and plotted with `plots.batch_summary()`.

**To load locations from a CSV file** (columns: name, lat, lon):
```python
import csv
with open('my_sites.csv') as f:
    BATCH_LOCATIONS = list(csv.DictReader(f))
```


In [ ]:
# ── Define batch locations ────────────────────────────────────────────────────
BATCH_LOCATIONS = [
    {'name': 'Apollo 15',    'lat': 26.1323, 'lon':  3.6285},
    {'name': 'Apollo 17',    'lat': 20.1911, 'lon': 30.7723},
    {'name': 'Equatorial 1', 'lat':  0.0,    'lon': 10.0   },
    {'name': 'Equatorial 2', 'lat':  0.0,    'lon': 45.0   },
    # Add more rows …
]

# ── Batch parameters ──────────────────────────────────────────────────────────
BATCH_SUNSCALE   = SUNSCALE
BATCH_ALBEDO     = ALBEDO
BATCH_EMISSIVITY = EMISSIVITY
BATCH_CHI        = CHI
BATCH_MODEL      = MODEL
BATCH_NDAYS      = NDAYS

print(f"Processing {len(BATCH_LOCATIONS)} location(s) …")
print(f"Model: {BATCH_MODEL}  SUNSCALE={BATCH_SUNSCALE}  ALBEDO={BATCH_ALBEDO}\n")

BATCH_RESULTS = analysis.run_batch(
    locations  = BATCH_LOCATIONS,
    elev_m     = ELEV_M,
    pixel_m    = PIXEL_M,
    map_res    = MAP_RES,
    z_grid     = Z_GRID,
    T_init     = T_INIT,
    chi        = BATCH_CHI,
    model_id   = models.MODEL_ID_MAP[BATCH_MODEL],
    sunscale   = BATCH_SUNSCALE,
    ndays      = BATCH_NDAYS,
    albedo     = BATCH_ALBEDO,
    emissivity = BATCH_EMISSIVITY,
)

# ── Summary table ─────────────────────────────────────────────────────────────
table = analysis.batch_to_table(BATCH_RESULTS)
print("\nResults summary:")
print(f"{'Name':<16} {'Lat':>7} {'Lon':>7} {'Elev':>7} {'T_min_0cm':>10} {'T_max_0cm':>10} {'T_mean_50cm':>12} {'RMSE':>7}")
print('─' * 85)
for i in range(len(table['name'])):
    rmse = table['RMSE_K'][i]
    rmse_str = f"{rmse:.3f}" if rmse is not None else "  —"
    print(f"{table['name'][i]:<16} {table['lat'][i]:>7.3f} {table['lon'][i]:>7.3f} "
          f"{table['elevation_m'][i]:>7.1f} "
          f"{table['T_min_0cm'][i]:>10.1f} {table['T_max_0cm'][i]:>10.1f} "
          f"{table['T_mean_50cm'][i]:>12.1f} {rmse_str:>7}")

# ── Batch plots ───────────────────────────────────────────────────────────────
fig = plots.batch_summary(BATCH_RESULTS, Z_GRID)
plt.show()
